# interpkit Quickstart

This notebook walks through the core operations available in interpkit.
We'll load GPT-2 and run scan, DLA, causal tracing, logit lens, attribution, patching, and more.

In [ ]:
import interpkit

model = interpkit.load("gpt2")

## Inspect

See the full module tree — types, parameter counts, shapes, and auto-detected roles (attention, MLP, norm, etc.).

In [ ]:
model.inspect()

## Scan — One-Command Overview

`scan()` runs DLA, logit lens, attention analysis, and attribution in one call, then surfaces the most interesting findings automatically. This is the fastest way to understand what a model is doing on an input.

In [ ]:
result = model.scan("The capital of France is")

## Direct Logit Attribution (DLA)

Why does the model predict a given token? DLA decomposes the output logit into contributions from each attention block and MLP, and breaks attention blocks down by individual head.

In [ ]:
result = model.dla("The capital of France is")

# The result is a dict — you can inspect it programmatically
print(f"Target token: {result['target_token']!r}")
print(f"Top contributor: {result['contributions'][0]['component']} "
      f"({result['contributions'][0]['logit_contribution']:+.3f})")
print(f"Top head: {result['head_contributions'][0]['component']} "
      f"({result['head_contributions'][0]['logit_contribution']:+.3f})")

## Logit Lens

Project each layer's hidden state into vocabulary space to see how the model's prediction evolves. By default, all token positions are analysed — producing a (layers x positions) heatmap. Pass `position=-1` to focus on just the last position.

In [ ]:
results = model.lens("The capital of France is")

## Causal Tracing

Which modules are most responsible for the model knowing that Paris is the capital of France?

We compare a clean prompt against a corrupted one and measure how much restoring each module's activation recovers the original answer.

In [ ]:
results = model.trace(
    "The capital of France is",
    "The capital of Poland is",
    top_k=15,
)

## Attribution

Gradient saliency — which input tokens matter most for the model's prediction?

In [ ]:
result = model.attribute("The capital of France is")

# attribute() now returns data programmatically
print(f"Tokens: {result['tokens']}")
print(f"Scores: {[f'{s:.2f}' for s in result['scores']]}")

## Activation Patching

Swap a single module's output from the clean run into the corrupted run. A high effect means that module is critical for the factual recall.

In [ ]:
result = model.patch(
    "The capital of France is",
    "The capital of Poland is",
    at="transformer.h.8.mlp",
)
print(f"Patch effect: {result['effect']:.4f}")

## Head-Level and Position-Level Patching

`patch()` also supports targeting a specific attention head or specific token positions.

In [ ]:
# Patch only attention head 3 in layer 8
result = model.patch(
    "The capital of France is",
    "The capital of Poland is",
    at="transformer.h.8",
    head=3,
)
print(f"Head-3 patch effect: {result['effect']:.4f}")

# Patch only specific token positions
result = model.patch(
    "The capital of France is",
    "The capital of Poland is",
    at="transformer.h.8.mlp",
    positions=[3, 4],
)
print(f"Position [3,4] patch effect: {result['effect']:.4f}")

## Ablation

Zero out a module's activations and measure how much the output changes.

In [ ]:
result = model.ablate(
    "The capital of France is",
    at="transformer.h.8.mlp",
    method="zero",
)
print(f"Ablation effect: {result['effect']:.4f}")

## Raw Activations

Extract activation tensors at any named module for your own analysis.

In [ ]:
act = model.activations("The capital of France is", at="transformer.h.8.mlp")
print(f"Shape: {act.shape}")
print(f"Norm:  {act.float().norm():.2f}")